In [1]:
import sys, os
# make godel_rest/ (the parent of user_scripts/) importable
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from server import ratio   # the module behind `server.cli ratio`
import requests
from itertools import combinations

In [2]:
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
tables = pd.read_html(StringIO(html))
tickers = tables[0]["Symbol"].tolist()
tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVB',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK.B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BNY',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF.B',
 'BLDR',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CASY',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CIEN',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COHR',
 'COIN',
 'CL',
 'CMCSA',
 'FIX',
 

In [3]:
from server import prices
res = prices.historical_prices("AMD", resolution="1D", months=6)
df = pd.DataFrame(res["bars"])
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df

,time,open,high,low,close,volume
date,,,,,,
2026-01-05,1767571200,230.245,234.020,220.480,221.08,3.118242e+07
2026-01-06,1767657600,222.710,222.920,211.250,214.35,3.976810e+07
2026-01-07,1767744000,212.125,212.125,207.170,210.02,2.936574e+07
2026-01-08,1767830400,210.900,210.940,203.330,204.68,2.696364e+07
2026-01-09,1767916800,205.720,207.300,203.070,203.17,2.403453e+07
...,...,...,...,...,...,...
2026-06-24,1782259200,520.820,524.960,503.500,519.74,2.552334e+07
2026-06-25,1782345600,543.930,550.880,507.000,532.57,2.651279e+07
2026-06-26,1782432000,519.795,525.110,502.610,521.58,5.160587e+07


In [4]:
#tickers = tickers[:10]
len(tickers)

503

In [5]:
# ---- 1. Fetch all closes once into a panel ----
def close_series(ticker, months=12):
    try:
        res = prices.historical_prices(ticker, resolution="1D", months=months)
        df = pd.DataFrame(res["bars"])
        df["date"] = pd.to_datetime(df["date"])
        return df.set_index("date")["close"].rename(ticker)
    except Exception as e:
        print(f"skip {ticker}: {e}")
        return None

series = [s for s in (close_series(t) for t in tickers) if s is not None]
panel = pd.concat(series, axis=1)          # dates × tickers
panel = panel.dropna(axis=1, thresh=int(0.9 * len(panel)))  # drop sparse tickers

# ---- 2. Returns + correlation matrix (one vectorized op) ----
returns = panel.pct_change().dropna(how="all")
corr = returns.corr()    

<positron-console-cell-5>:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.


In [6]:
cols = corr.columns
pairs = []
for a, b in combinations(cols, 2):
    r = corr.loc[a, b]
    if pd.notna(r):
        pairs.append((a, b, r))

ranked = pd.DataFrame(pairs, columns=["y", "x", "corr"])
ranked["abscorr"] = ranked["corr"].abs()
ranked = ranked.sort_values("abscorr", ascending=False).reset_index(drop=True)
ranked.head(50)

,y,x,corr,abscorr
0,GOOGL,GOOG,0.996828,0.996828
1,FOXA,FOX,0.984687,0.984687
2,NWSA,NWS,0.942740,0.942740
3,CPT,MAA,0.923705,0.923705
4,DHI,PHM,0.921834,0.921834
5,MLM,VMC,0.899608,0.899608
6,CFG,KEY,0.899321,0.899321
7,DAL,UAL,0.891640,0.891640
8,FITB,HBAN,0.885070,0.885070
9,STX,WDC,0.881277,0.881277


In [14]:
#ranked = ranked.drop([0,1,2])
ranked.reset_index(inplace=True, drop=True)
ranked.head()

,y,x,corr,abscorr
0,CPT,MAA,0.923705,0.923705
1,DHI,PHM,0.921834,0.921834
2,MLM,VMC,0.899608,0.899608
3,CFG,KEY,0.899321,0.899321
4,DAL,UAL,0.891640,0.891640


In [7]:
def ratio_row(y, x, months=6, window=120):
    d = ratio.ratio_analysis(y, x, months=months, correlation_window=window)
    row = {"y": d["y"]["ticker"], "x": d["x"]["ticker"],
           "months": months, "window": window}
    row.update(d["regression"])                       # beta, alpha, r, rsquared, std errs, t/p...
    row.update({f"corr_{k}": v for k, v in d["correlation"].items()})
    row.update({f"ratio_{k}": v for k, v in d["ratio"].items()})
    row["ylast"], row["xlast"] = d["ylast"], d["xlast"]
    return row

row = ratio_row("AMD", "SOX", months=6)
df = pd.DataFrame([row])
df

,y,x,months,window,beta,adjustedBeta,alpha,r,stdDevError,stdErrorAlpha,...,ttest,corr_last,corr_low,corr_high,corr_window,ratio_last,ratio_low,ratio_high,ylast,xlast
0,AMD,SOX,6,120,1.18442,1.123561,0.174599,0.766044,3.001932,0.277625,...,13.000467,0.767266,0.546718,0.771151,120,0.039351,0.023936,0.040384,539.49,13709.659996


In [16]:
#save panel, returns, and ranked as pickles for later use
panel.to_pickle("panel.pkl")
returns.to_pickle("returns.pkl")
ranked.to_pickle("ranked.pkl")
ranked.to_csv("ranked.csv", index=False)